# Lia: distância de edição para antigolpe e busca tolerante a erro

**Global Solution 2026.2 · CitizenTech · Dynamic Programming · Equipe AuxiLia**

| Integrante | RM |
|---|---|
| João Victor Melo Santos | 566640 |
| Murilo Jeronimo Ferreira Nunes | 560641 |
| Vinicius Kozonoe Guaglini | 567264 |
| Yan Lucas Gonçalves da Silva | 567046 |

## 1. O problema

A lia é um assistente de voz para pessoas com pouca prática de leitura e de cecular, em Taiobeiras (MG). Duas situações do dia a dia dela pedem o mesmo algoritmo:
1. **Golpe por link parecido.** A pessoa recebe um link como `g0v.br` ou `caixa-gov.br`, quase igual ao endereço oficial. Para quem lê com dificuldade, a diferença de uma letra não aparece.
2. **Pedido com erro.** A pessoa fala "benefisio do inss" ou "agendar consuta", e o texto chega com erro. A Lia precisa entender mesmo assim.

Nos dois casos, a pergunta é a mesma: **quão parecidos são dois textos?** A resposta é a distância de edição (Levenshtein), calculada com programação dinâmica.

## 2. Os dados

- `dados/dominios_oficiais.csv`: endereços oficiais de órgãos públicos, bancos e serviços. São dados públicos, conferidos nos sites oficiais em 16/11/2026.
- `dados/jeitos_de_pedir.csv`: frases que as pessoas usam para pedir cada ação da Lia. São dados simulados pela equipe. Não há nenhum dado pessoal.

In [1]:
import csv
from zoneinfo import reset_tzpath

import unicodedata

def carregar(caminho):
    """Lê um arquivo CSV e devolve uma lista de dicionários, um por linha."""
    with open(caminho, encoding="utf-8") as arquivo:
        return list(csv.DictReader(arquivo))

def normalizar(texto):
    """Deixa o texto em minúsculas, sem espaços nas pontas e sem acentos."""
    texto = texto.lower().strip()
    separado = unicodedata.normalize('NFD', texto)
    return "".join(letra for letra in separado if unicodedata.category(letra) != "Mn")

dominios = carregar("dados/dominios_oficiais.csv")
jeitos = carregar("dados/jeitos_de_pedir.csv")

print(len(dominios), "domínios oficiais")
print(len(jeitos), "jeitos de pedir")
print(dominios[0])
print(normalizar("  Benefício do INSS "))

15 domínios oficiais
23 jeitos de pedir
{'dominio': 'gov.br', 'servico': 'Portal gov.br', 'tipo': 'governo'}
beneficio do inss


## 3. O algoritmo: distância de Levenshtein com programação dinâmica

A distância de Levenshtein entre dois textos é o menor número de operações para um virar o outro. As operações são trocar, inserir ou apagar uma letra.

**Estrutura de dados:** uma tabela `t` com `(m + 1)` linhas e `(n + 1)` colunas, em que `m` e `n` são os tamanhos dos textos. A casa `t[i][j]` guarda a distância entre as `i` primeiras letras de `a` e as `j` primeiras letras de `b`.

**Regra de preenchimento:**

- `t[i][0] = i` e `t[0][j] = j` (comparação com o texto vazio);
- `t[i][j] = min(t[i-1][j] + 1, t[i][j-1] + 1, t[i-1][j-1] + custo)`, com `custo = 0` se `a[i-1] == b[j-1]` e `custo = 1` se não.

Cada casa é calculada uma vez só, a partir de três casas já prontas. É isso que faz ser programação dinâmica: os subproblemas se repetem, e a tabela evita recalcular.

In [2]:
def tabela_de_distancias(a, b):
    """Monta a tabela de PD: t[i][j] é a distância entre as i primeiras letras de a e as j primeiras de b."""
    m, n = len(a), len(b)
    t = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        t[i][0] = i
    for j in range(n + 1):
        t[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            custo = 0 if a[i - 1] == b[j - 1] else 1
            t[i][j] = min(
                t[i - 1][j] + 1,
                t[i][j - 1] + 1,
                t[i - 1][j - 1] + custo
            )
    return t

def distancia(a, b):
    """Distância de Levenshtein: o menor número de trocas, inserções e remoções para a virar b."""
    t = tabela_de_distancias(a, b)
    return t[len(a)][len(b)]

def mostrar_tabela(a, b):
    """Desenha a tabela de PD, com as letras de a nas linhas e as de b nas colunas."""
    t = tabela_de_distancias(a, b)
    print("      " + "  ".join(b))
    for i, linha in enumerate(t):
        letra = a[i - 1] if i > 0 else " "
        print(letra, " ".join(f"{valor:2}" for valor in linha))

mostrar_tabela("casa", "caso")

      c  a  s  o
   0  1  2  3  4
c  1  0  1  2  3
a  2  1  0  1  2
s  3  2  1  0  1
a  4  3  2  1  1


In [3]:
testes = [
    ("casa", "casa", 0),
    ("casa", "caso", 1),
    ("gato", "pato", 1),
    ("inss", "ins", 1),
    ("", "abc", 3),
    ("beneficio", "benefisio", 1),
    ("kitten", "sitting", 3),
]

for a, b, esperado in testes:
    obtido = distancia(a, b)
    assert obtido == esperado, f"{a} -> {b}: esperado {esperado}, obtido {obtido}"
    print (f"{a!r} -> {b!r}: {obtido}")

print ("Todos os testes passaram.")

'casa' -> 'casa': 0
'casa' -> 'caso': 1
'gato' -> 'pato': 1
'inss' -> 'ins': 1
'' -> 'abc': 3
'beneficio' -> 'benefisio': 1
'kitten' -> 'sitting': 3
Todos os testes passaram.


## 4. Eficiência: três jeitos de calcular a mesma distância

A mesma regra pode ser calculada de três formas. As três dão o mesmo resultado, e o que muda é o tempo e a memória.

- **Ingênua:** recursão sem guardar nada. Recalcula os mesmos pedaços muitas vezes.
- **Tabela:** a versão da seção 3. Calcula cada classe de uma vez só.
- **Duas linhas:** a mesma conta da tabela, guardando só a linha anterior e a atual.

In [4]:
def distancia_ingenua(a, b):
    """Mesma regra da tabela, mas sem guardar nada: recalcula os mesmos pedaços muitas vezes."""
    if len(a) == 0:
        return len(b)
    if len(b) == 0:
        return len(a)
    custo = 0 if a[-1] == b[-1] else 1
    return min(
        distancia_ingenua(a[:-1], b) + 1,
        distancia_ingenua(a, b[:-1]) + 1,
        distancia_ingenua(a[:-1], b[:-1]) + custo,
    )

In [5]:
def distancia_duas_linhas(a, b):
    """Mesma conta da tabela, mas guardando só duas linhas: a anterior e a atual."""
    if len(a) < len(b):
        a, b = b, a
    anterior = list(range(len(b) + 1))
    for i in range(1, len(a) + 1):
        atual = [i] + [0] * len(b)
        for j in range(1, len(b) + 1):
            custo = 0 if a[i - 1] == b[j - 1] else 1
            atual[j] = min(
                anterior[j] + 1,
                atual[j - 1] + 1,
                anterior[j - 1] + custo,
            )
        anterior = atual
    return anterior[len(b)]

In [6]:
for a, b, esperado in testes:
    assert distancia_ingenua(a, b) == esperado, f"ingênua: {a} -> {b}"
    assert distancia_duas_linhas(a, b) == esperado, f"duas linhas: {a} -> {b}"

print("As três versões dão o mesmo resultado nos", len(testes), "testes.")

As três versões dão o mesmo resultado nos 7 testes.


In [7]:
import time

def medir(funcao, a, b):
    """Devolve o tempo, em segundos, que a função leva para comparar a e b."""
    inicio = time.perf_counter()
    funcao(a, b)
    return time.perf_counter() - inicio


print(f"{'letras':>6} {'ingênua':>12} {'tabela':>12} {'duas linhas':>12}")
for tamanho in [4, 6, 8, 10]:
    a, b = "a" * tamanho, "b" * tamanho
    print(f"{tamanho:>6} {medir(distancia_ingenua, a, b):>11.5f}s {medir(distancia, a, b):>11.5f}s {medir(distancia_duas_linhas, a, b):>11.5f}s")

print()
print(f"{'letras':>6} {'tabela':>12} {'duas linhas':>12} {'casas da tabela':>16} {'casas em 2 linhas':>18}")
for tamanho in [10, 100, 1000]:
    a, b = "a" * tamanho, "b" * tamanho
    print(f"{tamanho:>6} {medir(distancia, a, b):>11.5f}s {medir(distancia_duas_linhas, a, b):>11.5f}s {(tamanho + 1) ** 2:>16} {2 * (tamanho + 1):>18}")

letras      ingênua       tabela  duas linhas
     4     0.00011s     0.00001s     0.00001s
     6     0.00170s     0.00001s     0.00001s
     8     0.04861s     0.00002s     0.00001s
    10     1.46374s     0.00003s     0.00002s

letras       tabela  duas linhas  casas da tabela  casas em 2 linhas
    10     0.00002s     0.00002s              121                 22
   100     0.00174s     0.00130s            10201                202
  1000     0.22260s     0.16353s          1002001               2002


### O que as medições mostram

| Versão | Tempo | Memória |
|---|---|---|
| Ingênua | Exponencial: cada chamada abre até três outras | Pequena |
| Tabela | `O(m × n)`: uma conta por casa | `O(m × n)`: a tabela inteira |
| Duas linhas | `O(m × n)`: a mesma conta | `O(min(m, n))`: só duas linhas |

- **A versão ingênua não serve para a Lia.** A cada 2 letras a mais, ela fica cerca de 30 vezes mais lenta. Com 10 letras, já leva cerca de 1,5 segundo, e as versões com programação dinâmica levam centésimos de milésimo de segundo. Um endereço como `cadunico.dataprev.gov.br` tem 24 letras.
- **A tabela e a versão de duas linhas levam quase o mesmo tempo,** porque fazem a mesma conta. A diferença está na memória: com 1000 letras, a tabela guarda cerca de 1 milhão de casas, e a versão de duas linhas guarda cerca de 2 mil.
- **Na Lia, usamos a versão de duas linhas** nas aplicações, porque só precisamos do número final. A tabela completa fica para explicar o algoritmo.

## 5. Aplicação 1: antigolpe

A Lia compara o endereço de um link com os endereços oficiais que ela conhece e dá um de três vereditos:

- **OFICIAL:** é igual a um endereço oficial, ou é um subdomínio dele (por exemplo, `meu.inss.gov.br` termina em `.gov.br`).
- **IMITAÇÃO:** não é oficial, mas a distância de edição para um oficial é pequena. É o sinal de golpe.
- **DESCONHECIDO:** não se parece com nenhum oficial. A Lia não acusa, mas pede cuidado.

O limite de imitação é de 1 letra de diferença a cada 5 letras do endereço oficial.

In [11]:
from urllib.parse import urlparse

def dominio_de(link):
    """Tira o domínio de um link: de "https://www.gov.br/inss" sobra "gov.br"."""
    if "://" not in link:
        link = "https://" + link
    dominio = (urlparse(link).hostname or "").lower()
    return dominio[4:] if dominio.startswith("www.") else dominio

def oficial_de(dominio):
    """Devolve o registro oficial se o domínio for igual a um oficial ou um subdomínio dele. Senão, devolve None."""
    encontrados = [o for o in dominios if dominio == o["dominio"] or dominio.endswith("." + o["dominio"])]
    if not encontrados:
        return None
    return max(encontrados, key=lambda o: len(o["dominio"]))

def limite_de_erros(oficial):
    """Quantas letras de diferença ainda contam como imitação: 1 a cada 5 letras do endereço oficial."""
    return max(1, len(oficial) // 5)

def ranking_de_oficiais(dominio):
    """Ordena os domínios oficiais do mais parecido para o menos parecido com o domínio recebido."""
    candidatos = [(distancia_duas_linhas(dominio, oficial["dominio"]), oficial) for oficial in dominios]
    return sorted(candidatos, key=lambda par: par[0])

def analisar_link(link):
    """Diz se o link é oficial, imitação de um oficial ou desconhecido."""
    dominio = dominio_de(link)

    oficial = oficial_de(dominio)
    if oficial is not None:
        return {"dominio": dominio, "veredito": "OFICIAL", "parecido_com": oficial["servico"], "distancia": 0}

    menor_distancia, mais_parecido = ranking_de_oficiais(dominio)[0]
    if menor_distancia <= limite_de_erros(mais_parecido["dominio"]):
        veredito = "IMITACAO"
    else:
        veredito = "DESCONHECIDO"
    return {"dominio": dominio, "veredito": veredito, "parecido_com": mais_parecido["servico"], "distancia": menor_distancia}

In [12]:
print("Os 3 oficiais mais parecidos com g0v.br:")
for d, oficial in ranking_de_oficiais("g0v.br")[:3]:
    print(f"  {oficial['dominio']:<12} distância {d}")
print()

links_de_teste = [
    ("https://www.gov.br/inss", "OFICIAL"),
    ("https://meu.inss.gov.br/central", "OFICIAL"),
    ("https://g0v.br/login", "IMITACAO"),
    ("http://caixa-gov.br/fgts", "IMITACAO"),
    ("http://meuinss-gov.br", "IMITACAO"),
    ("http://bradesc0.com.br", "IMITACAO"),
    ("https://cemlg.com.br/2via", "IMITACAO"),
    ("http://correlos.com.br/rastreio", "IMITACAO"),
    ("https://youtube.com", "DESCONHECIDO"),
    ("https://oi.com.br", "DESCONHECIDO"),
]

acertos = 0
print(f"{'link':<34} {'veredito':<13} {'parecido com':<26} distancia")
for link, esperado in links_de_teste:
    resultado = analisar_link(link)
    acertos += resultado["veredito"] == esperado
    print(f"{link:<34} {resultado['veredito']:<13} {resultado['parecido_com']:<26} {resultado['distancia']}")

print(f"\nAcertos: {acertos} de {len(links_de_teste)}")

Os 3 oficiais mais parecidos com g0v.br:
  gov.br       distância 1
  mg.gov.br    distância 4
  bcb.gov.br   distância 5

link                               veredito      parecido com               distancia
https://www.gov.br/inss            OFICIAL       Portal gov.br              0
https://meu.inss.gov.br/central    OFICIAL       Meu INSS                   0
https://g0v.br/login               IMITACAO      Portal gov.br              1
http://caixa-gov.br/fgts           IMITACAO      Caixa Econômica Federal    1
http://meuinss-gov.br              IMITACAO      Meu INSS                   2
http://bradesc0.com.br             IMITACAO      Bradesco                   1
https://cemlg.com.br/2via          IMITACAO      Cemig (conta de luz)       1
http://correlos.com.br/rastreio    IMITACAO      Correios                   1
https://youtube.com                DESCONHECIDO  Portal gov.br              9
https://oi.com.br                  DESCONHECIDO  Banco do Brasil            2

Acertos: 1

In [13]:
FALAS = {
    "OFICIAL": "Pode confiar. Esse é o endereço oficial: {servico}.",
    "IMITACAO": "Cuidado! Esse endereço imita o oficial ({servico}), mas não é ele. Pode ser golpe. Não clique e não digite a sua senha.",
    "DESCONHECIDO": "Eu não conheço esse endereço. Não digite senha nem dados pessoais nele antes de confirmar com alguém de confiança.",
}

def fala_da_lia(resultado):
    """Monta a frase que a Lia fala para a pessoa."""
    return FALAS[resultado["veredito"]].format(servico=resultado["parecido_com"])

for link in ["https://g0v.br/login", "https://meu.inss.gov.br/central", "https://youtube.com"]:
    print(link)
    print("  Lia:", fala_da_lia(analisar_link(link)))

https://g0v.br/login
  Lia: Cuidado! Esse endereço imita o oficial (Portal gov.br), mas não é ele. Pode ser golpe. Não clique e não digite a sua senha.
https://meu.inss.gov.br/central
  Lia: Pode confiar. Esse é o endereço oficial: Meu INSS.
https://youtube.com
  Lia: Eu não conheço esse endereço. Não digite senha nem dados pessoais nele antes de confirmar com alguém de confiança.


## 6. Aplicação 2: busca tolerante a erro

A pessoa fala, e o texto pode chegar com erro ("benefisio do inss", "avisa meu fio"). A Lia compara o pedido com cada frase de `jeitos_de_pedir.csv` e fica com a mais parecida.

Como as frases têm tamanhos muito diferentes, a comparação usa uma nota de semelhança de 0 a 1:

`semelhança = 1 - distância / tamanho do maior texto`

Os textos são normalizados antes (minúsculas, sem acentos), para que "Benefício" e "beneficio" contem como iguais.

Se a melhor nota for menor que 0,5, a Lia não arrisca: ela pede para a pessoa falar de outro jeito. Quando ela entende, o tipo da ação (por exemplo, `avisar_familiar`) é o mesmo do contrato com o núcleo Java, que decide a confirmação.

Custo: cada pedido é comparado com todas as `k` frases, então a busca custa `O(k × m × n)`. Com 23 frases curtas, isso leva milissegundos.

In [20]:
LIMITE_SEMELHANCA = 0.5

def semelhanca(a, b):
    """Nota de 0 a 1: 1 quer dizer textos iguais, 0 quer dizer totalmente diferentes."""
    maior = max(len(a), len(b))
    if maior == 0:
        return 1.0
    return 1 - distancia_duas_linhas(a, b) / maior

def buscar(pedido, quantos=3):
    """Devolve os jeitos de pedir mais parecidos com o pedido, do mais parecido para o menos parecido."""
    pedido = normalizar(pedido)
    candidatos = [(semelhanca(pedido, normalizar(jeito["frase"])), jeito) for jeito in jeitos]
    candidatos.sort(key=lambda par: par[0], reverse=True)
    return candidatos[:quantos]

def entender(pedido):
    """Descobre qual ação a pessoa pediu. Devolve None quando nenhuma frase é parecida o bastante."""
    nota, jeito = buscar(pedido, 1)[0]
    if nota >= LIMITE_SEMELHANCA:
        return jeito ["acao"]
    return None

In [21]:
print('Os 3 jeitos de pedir mais parecidos com "benefisio do inss":')
for nota, jeito in buscar("benefisio do inss"):
    print(f"  {jeito['frase']:<22} {jeito['acao']:<20} semelhança {nota:.2f}")
print()

pedidos_de_testes = [
    ("benefisio do inss", "consultar_beneficio"),
    ("agendar consuta", "agendar_por_voz"),
    ("marca medico", "agendar_por_voz"),
    ("avisa meu fio", "avisar_familiar"),
    ("paga boleto", "pagar_boleto"),
    ("pagar conta de lus", "pagar_boleto"),
    ("ler esa carta", "tirar_foto"),
    ("ver minha multa", "guiar"),
    ("abri o zap", "guiar_app"),
    ("manda aquilo pra ela", None),
    ("bom dia", None),
    ("quero dançar forró", None),
]

acertos = 0
print(f"{'pedido':<22} {'ação entendida':<20} {'frase mais parecida':<32} semelhança")
for pedido, esperado in pedidos_de_testes:
    nota, jeito = buscar(pedido, 1)[0]
    acao = entender(pedido)
    acertos += acao == esperado
    print(f"{pedido:<22} {str(acao):<20} {jeito['frase']:<32} {nota:.2f}")

print(f"\nAcertos: {acertos} de {len(pedidos_de_testes)}")

Os 3 jeitos de pedir mais parecidos com "benefisio do inss":
  benefício do inss      consultar_beneficio  semelhança 0.94
  quanto caiu do inss    consultar_beneficio  semelhança 0.53
  tirar foto do papel    tirar_foto           semelhança 0.26

pedido                 ação entendida       frase mais parecida              semelhança
benefisio do inss      consultar_beneficio  benefício do inss                0.94
agendar consuta        agendar_por_voz      agendar consulta                 0.94
marca medico           agendar_por_voz      marcar médico                    0.92
avisa meu fio          avisar_familiar      avisar meu filho                 0.81
paga boleto            pagar_boleto         pagar boleto                     0.92
pagar conta de lus     pagar_boleto         pagar conta de luz               0.94
ler esa carta          tirar_foto           ler essa carta                   0.93
ver minha multa        guiar                ver minhas multas                0.88
abri o z

In [22]:
for pedido in ["avisa meu fio", "manda aquilo pra ela"]:
    acao = entender(pedido)
    print(f"Pessoa: {pedido}")
    if acao is None:
        print("  Lia: Não entendi direito. O que você quer fazer, e para quem?")
    else:
        print(f"  Lia entendeu a ação {acao}, que segue para o núcleo Java decidir a confirmação.")

Pessoa: avisa meu fio
  Lia entendeu a ação avisar_familiar, que segue para o núcleo Java decidir a confirmação.
Pessoa: manda aquilo pra ela
  Lia: Não entendi direito. O que você quer fazer, e para quem?


## 7. Conclusões

- **O mesmo algoritmo resolve os dois problemas da Lia.** A distância de Levenshtein encontrou as 8 imitações de endereços oficiais dos testes (`g0v.br`, `bradesc0.com.br`, `cemlg.com.br`...) sem acusar endereços legítimos, e entendeu 9 pedidos com erro sem confundir os 3 que não tinham sentido.
- **A programação dinâmica é o que torna isso possível.** A versão ingênua leva mais de um segundo com só 10 letras. A tabela calcula cada casa uma vez, em `O(m × n)`, e a versão de duas linhas faz a mesma conta com memória `O(min(m, n))`.
- **A Lia não acusa sem motivo.** Endereços que não se parecem com nenhum oficial são marcados como desconhecidos, com um pedido de cuidado. Pedidos sem sentido fazem a Lia perguntar de novo, em vez de adivinhar.

**Limitações:**

- Os limites (1 letra a cada 5 no antigolpe e 0,5 de semelhança na busca) foram escolhidos a partir dos exemplos do notebook. Com dados reais, eles precisariam ser ajustados.
- A distância compara letras, e não sentidos: "zap" e "whatsapp" só ficaram parecidos porque o resto da frase ajudou.
- As frases de `jeitos_de_pedir.csv` são simuladas pela equipe. Um próximo passo é coletá-las com as pessoas atendidas no CRAS Planalto.